### Token Length Profiling

This notebook complements `01_data_profiling_tweet_lengths.ipynb` by measuring subword token lengths using the ModernGBERT tokenizer.

The goal here is to determine the effective sequence length used during training and to verify how many tweets, if any, are truncated at that limit. ModernGBERT's architecture supports up to 8,192 tokens. (We use 1024 in training)

---

##### <b>Imports</b>

In [19]:
import sys
import json
import statistics
from pathlib import Path
from rich.console import Console

# Hugging Face Imports
from transformers import AutoProcessor

console = Console()

# Path to this notebook
notebook_dir = Path.cwd()

# Project root directory
project_root_dir = notebook_dir.parent.parent.parent

# Source path
src_path = project_root_dir / "src"
sys.path.append(str(src_path))

console.print(f"Project root: {project_root_dir}", style="cyan")
console.print(f"Source path: {src_path}", style="cyan")
console.print(f"Source path exists: {src_path.exists()}", style="cyan")

# Local imports
import data_utils
import config_utils

Project root: /home/samuel/VSCodeProjects/Bachelors-Thesis/Competition-Solution

Source path: /home/samuel/VSCodeProjects/Bachelors-Thesis/Competition-Solution/src

Source path exists: True

##### <b>Loading the data</b>

In [20]:
# Loads training and test data for all subtasks
df_call2action_train, df_dbo_train, df_violence_train = data_utils.load_competition_data(data_dir="../../../data/raw/", type="train")
df_call2action_test, df_dbo_test, df_violence_test = data_utils.load_competition_data(data_dir="../../../data/raw/", type="test")

Loading competition data...

Data of type 'train' loaded successfully.

Loading competition data...

Data of type 'test' loaded successfully.

##### <b>Loading the config</b>

In [21]:
# Path to the configs directory
config_base_dir_nb = project_root_dir / "configs"

# Defines the paths to the global, subtask and experiment configs
base_cfg_path = config_base_dir_nb / "base.yaml" # Global base config
subtask_base_cfg_path = config_base_dir_nb / "subtask1_call2action" / "base.yaml" # Subtask base config
experiment_cfg_path = config_base_dir_nb / "subtask1_call2action" / "ModernGBERT_baseline.yaml" # Experiment config

# Loads the global, subtask and experiment configs
cfg = config_utils.load_config(
    base_config_path=base_cfg_path,
    subtask_config_path=subtask_base_cfg_path,
    experiment_config_path=experiment_cfg_path
)

# Raises an error if one of the config files is not found
if not cfg:
    raise ValueError(f"Configuration files could not be loaded. Please check the config YAML files in the \"configs\" directory:\n"
                     f"    Global base config: {base_cfg_path}\n"
                     f"    Subtask base config: {subtask_base_cfg_path}\n"
                     f"    Experiment config: {experiment_cfg_path}")

console.print(f"Successfully loaded and merged configurations for: {cfg['model_checkpoint']}", style="green")

Successfully loaded and merged configurations for: LSX-UniWue/ModernGBERT_134M

##### <b>Loading the processor</b>

In [22]:
# Model configuration and paths
console.print(f"Model: {cfg['model_checkpoint']}", style="bold")

# Checks for local model first, fallback to HF Hub
local_model_path = project_root_dir / cfg['paths']['base_models_dir_name'] / cfg['model_checkpoint']
if local_model_path.exists():
    model_path = str(local_model_path)
    console.print(f"Found local model at: {model_path}", style="cyan")
else:
    model_path = cfg['model_checkpoint']
    console.print(f"Local model not found, using Hugging Face Hub: {model_path}", style="yellow")

console.print(f"Loading AutoProcessor for '{model_path}'", style="green")

# Loads the processor and adds the special tokens (Anonymization tokens that we set during the data preprocessing phase)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
processor.add_special_tokens({"additional_special_tokens": cfg['tokenization']['special_tokens']})

# 1024
EFFECTIVE_MAX_LENGTH = processor.model_max_length

# The model architecture supports longer sequences (max_position_embeddings from config.json)
model_config_path = Path(model_path) / "config.json"
with open(model_config_path, "r") as config_file:
    model_config = json.load(config_file)
ARCHITECTURAL_MAX_LENGTH = model_config['max_position_embeddings']

console.print(f"Processor loaded with {len(cfg['tokenization']['special_tokens'])} special tokens, total: {len(processor)}", style="cyan")
console.print(f"Effective max length during training (processor.model_max_length): [bold]{EFFECTIVE_MAX_LENGTH:,}[/bold]")
console.print(f"Architectural max length (config.json max_position_embeddings): [bold]{ARCHITECTURAL_MAX_LENGTH:,}[/bold]")

Model: LSX-UniWue/ModernGBERT_134M

Found local model at: 
/home/samuel/VSCodeProjects/Bachelors-Thesis/Competition-Solution/models/base_models/LSX-UniWue/ModernGBERT_134M

Loading AutoProcessor for 
'/home/samuel/VSCodeProjects/Bachelors-Thesis/Competition-Solution/models/base_models/LSX-UniWue/ModernGBERT_134M'

Processor loaded with 6 special tokens, total: 31108

Effective max length during training (processor.model_max_length): 1,024

Architectural max length (config.json max_position_embeddings): 8,192

##### <b>Profiling subword token lengths</b>

In [23]:
# Tokenises all tweets per subtask (train + test combined) and computes token length statistics
all_lengths = {}
for subtask_name, df_train, df_test in [
    ("Call2Action", df_call2action_train, df_call2action_test),
    ("DBO", df_dbo_train, df_dbo_test),
    ("Violence", df_violence_train, df_violence_test),
]:
    train_texts = df_train['description'].dropna().tolist()
    test_texts = df_test['description'].dropna().tolist()
    combined_texts = train_texts + test_texts

    console.print(f"Tokenising {subtask_name} ({len(combined_texts):,} tweets)...", style="italic")

    token_lengths = []
    for text in combined_texts:
        encoded = processor.encode(text, add_special_tokens=True)
        token_lengths.append(len(encoded))

    all_lengths[subtask_name] = token_lengths

    # Computes the total tweets, max, mean, median, stdev, and coverage
    total_tweets = len(token_lengths)
    max_token_length = max(token_lengths)
    mean_token_length = statistics.mean(token_lengths)
    median_token_length = statistics.median(token_lengths)
    stdev_token_length = statistics.stdev(token_lengths)

    tweets_within_limit = 0
    for length in token_lengths:
        if length <= EFFECTIVE_MAX_LENGTH:
            tweets_within_limit += 1

    tweets_exceeding_limit = total_tweets - tweets_within_limit
    within_percentage = 100 * tweets_within_limit / total_tweets
    exceeding_percentage = 100 * tweets_exceeding_limit / total_tweets

    console.rule(f"[bold cyan]{subtask_name}[/bold cyan]")
    console.print(f"Total tweets: [bold]{total_tweets:>8,}[/bold]")
    console.print(f"Max token length: [bold]{max_token_length:>8,}[/bold]")
    console.print(f"Mean token length: [bold]{mean_token_length:>8.1f}[/bold]")
    console.print(f"Median: [bold]{median_token_length:>8.1f}[/bold]")
    console.print(f"Stdev: [bold]{stdev_token_length:>8.1f}[/bold]")
    console.print(f"Within {EFFECTIVE_MAX_LENGTH}: [bold green]{tweets_within_limit:>8,}[/bold green] ({within_percentage:.3f}%)")
    console.print(f"Exceeds {EFFECTIVE_MAX_LENGTH}: [bold red]{tweets_exceeding_limit:>8,}[/bold red] ({exceeding_percentage:.3f}%)")
    console.print()

Tokenising Call2Action (9,822 tweets)...

Token indices sequence length is longer than the specified maximum sequence length for this model (1477 > 1024). Running this sequence through the model will result in indexing errors


─────────────────────────────────────────────────── Call2Action ───────────────────────────────────────────────────

Total tweets:    9,822

Max token length:    2,444

...

Mean token length:     40.9

Median:     25.0

Stdev:     69.1

Within 1024:   11,110 (99.928%)

Exceeds 1024:        8 (0.072%)

In [24]:
# Displays the cumulative distribution across the subtasks
combined_lengths = []
for subtask_lengths in all_lengths.values():
    for length in subtask_lengths:
        combined_lengths.append(length)

total_combined = len(combined_lengths)
global_max_token_length = max(combined_lengths)

# Counts tweets truncated at the effective training limit
tweets_truncated = 0
for length in combined_lengths:
    if length > EFFECTIVE_MAX_LENGTH:
        tweets_truncated += 1

console.rule("[bold cyan]Cumulative Distribution (all subtasks combined)[/bold cyan]")
console.print(f"Total tweets (all subtasks): [bold]{total_combined:,}[/bold]")
console.print(f"Global max token length: [bold]{global_max_token_length:,}[/bold]")
console.print(f"Effective training limit (processor.model_max_length): [bold]{EFFECTIVE_MAX_LENGTH:,}[/bold]")
console.print(f"Tweets truncated during training: [bold red]{tweets_truncated}[/bold red] ({100 * tweets_truncated / total_combined:.2f}%)\n")

thresholds = [64, 128, 256, 512, 1024, 2048, 4096, 8192]
for threshold in thresholds:
    tweets_at_or_below = 0

    for length in combined_lengths:
        if length <= threshold:
            tweets_at_or_below += 1
    
    percentage = 100 * tweets_at_or_below / total_combined

    # Highlights the effective training limit
    if threshold == EFFECTIVE_MAX_LENGTH:
        console.print(f"<= {threshold:>5} tokens: [bold]{tweets_at_or_below:>6,}[/bold] ({percentage:.2f}%)")
    else:
        console.print(f"<= {threshold:>5} tokens: [bold]{tweets_at_or_below:>6,}[/bold] ({percentage:.2f}%)")

───────────────────────────────── Cumulative Distribution (all subtasks combined) ─────────────────────────────────

Total tweets (all subtasks): 31,588

Global max token length: 2,669

Effective training limit (processor.model_max_length): 1,024

Tweets truncated during training: 20 (0.06%)

...

<=   512 tokens: 31,528 (99.81%)

<=  1024 tokens: 31,568 (99.94%)

<=  2048 tokens: 31,584 (99.99%)

<=  4096 tokens: 31,588 (100.00%)

<=  8192 tokens: 31,588 (100.00%)

<u><p>Interpretation</p></u>

Although ModernGBERT's architecture supports sequences up to 8,192 tokens, we set the effective training limit to 1,024 for computational efficiency. Approximately 20 tweets (0.06%) exceed that threshold and are truncated during training. The distribution is heavily right-skewed, with 99.11% of tweets fitting within 256 tokens, so the practical impact of the 1,024 cutoff is negligible.